In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("data.csv")
df["observation_date"] = pd.to_datetime(df["observation_date"])
df = df.rename(columns={"observation_date": "date", "IPG2211A2N": "value"})
df = df.set_index("date").sort_index()
df

In [ ]:
from statsmodels.tsa.seasonal import STL

decomposition = STL(
    df["value"],
    period=48,
)
decomposition = decomposition.fit()

_, ax = plt.subplots(4, 1, figsize=(20, 8), sharex=True)

ax[0].plot(df["value"], color="#6B001E")
ax[0].set_title("Stromverbrauch")

ax[1].plot(decomposition.trend, color="#000000FF")
ax[1].set_title("Trend")

ax[2].plot(decomposition.seasonal, color="#D386FF")
ax[2].set_title("Saisonalität")

ax[3].plot(decomposition.resid, color="#9B0000")
ax[3].set_title("Residuum")

plt.margins(x=0)
plt.tight_layout()
plt.show()

# Residuum der gesamten Zeitreihe
plt.figure(figsize=(20, 8))

plt.plot(df.index, decomposition.resid, color="#9B0000", label="Index (Bereinigt/Residuum)")

plt.title("Energieverbrauch")
plt.xlabel("Datum")
plt.ylabel("Index")
plt.grid(alpha=0.5)
plt.margins(x=0)
plt.legend()
plt.tight_layout()
plt.show()

df["trend"] = decomposition.trend
df["trendbereinigt"] = df["value"] - df["trend"]
df["saison"] = decomposition.seasonal
df["residuum"] = decomposition.resid

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Autokorrelation
# TODO Lags festlegen / Begründung formulieren
_, ax = plt.subplots(1, 2, figsize=(20, 8))

plot_acf(df["residuum"], lags=48, ax=ax[0])
ax[0].set_title("Autokorrelation")

plot_pacf(df["residuum"], lags=48, ax=ax[1])
ax[1].set_title("Partielle Autokorrelation")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from statsmodels.tsa.arima.model import ARIMA

time_series = df["residuum"]

# Note: ARMA is a special case of ARIMA where d=0 (no differencing)
# The order (p,d,q) represents (AR order, Differencing order, MA order)
model = ARIMA(time_series, order=(2, 1, 2))  # TODO Werte festlegen
model_fit = model.fit()

print(model_fit.summary())

# Vorhersage
pred = model_fit.predict()

# Metriken
# Hier zu betrachten: MSE
print("MSE:", mean_squared_error(time_series, pred))
print("RMSE:", root_mean_squared_error(time_series, pred))
print("MAE:", mean_absolute_error(time_series, pred))
print("R2:", r2_score(time_series, pred))

# Darstellung
plt.figure(figsize=(20, 8))

plt.plot(df.index, df["residuum"], color="#9B0000", alpha=0.2, label="Index (Residuum)")
plt.plot(df.index, pred, color="#9B0000", label="Index (Vorhersage)")

plt.title("Energieverbrauch")
plt.xlabel("Datum")
plt.ylabel("Index")
plt.grid(alpha=0.5)
plt.margins(x=0)
plt.legend()
plt.tight_layout()
plt.show()